# Medical Prescription OCR LLM comprehension Evaluation with DeepEval

Runs mass evaluation on mass amount of ligh Ollama models (<8b parameters) in order to chose the best model for our task.

**Pipeline:**
```
Raw OCR text → Ollama model → JSON output → DeepEval scoring
```

**Entities extracted:** patient information, medical facility, date, doctor information, medications (name, dosage, form, quantity, posologie (frequency), QSP (duration))


In [ ]:
# Install required packages
%pip install deepeval rich pandas tabulate --quiet
%sudo apt-get install zstd && !sudo apt install pciutils
%curl -fsSL https://ollama.com/install.sh | sh
%pip install gdown --quiet

import os 
import gdown
import pathlib

---
## 2. Test Dataset — OCR Examples with Ground Truth

In [2]:
#----------------------------------------------------------
# DOWNLOADING THE PRESCRIPTION PICTURES FROM GOOGLE DRIVE
# ---------------------------------------------------------

# The folder URL or ID from Google Drive
url = "https://drive.google.com/drive/folders/16FthvW2bPYRqt9PyrMI36Mmgruepr_ap?usp=drive_link"

# Downloads the entire folder and its contents
gdown.download_folder(url, quiet=True, use_cookies=False)

!ls # making sure that 'tier_payant_ordonnances'folder is downloaded and present in the current directory 

BASE_PRESCRIPTION_FOLDER = pathlib.Path("tier_payant_ordonnances")

prescriptions_files_paths = list(BASE_PRESCRIPTION_FOLDER.glob("*recto.jpg"))

print("Total number of prescription files:", len(prescriptions_files_paths))
print("---")

for path in prescriptions_files_paths:
    print(path)


In [3]:
!python -m pip install paddlepaddle==3.2.0 -i https://www.paddlepaddle.org.cn/packages/stable/cpu/
!python -c "import paddle; print(paddle.__version__)"
!python -m pip install "transformers>=5.4.0"
!python -m pip install "paddleocr[all]"
!paddleocr -h

In [4]:
import cv2

def apply_binarization(input_path, threshold_value=127, use_otsu=False):
    
    image = cv2.imread(input_path, cv2.IMREAD_GRAYSCALE)
    
    if image is None:
        raise ValueError(f"Could not open or find the image at {input_path}")
    
    if use_otsu:
        _, binary_image = cv2.threshold(image, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    else:
        _, binary_image = cv2.threshold(image, threshold_value, 255, cv2.THRESH_BINARY)

    return binary_image  

def preprocessing_image(input_image_path, output_image_path, binar_params, save_output=True):
    binary_image = apply_binarization(input_image_path, binar_params.get("threshold_value", 127), binar_params.get("use_otsu", False))
    if save_output:
        cv2.imwrite(output_image_path, binary_image)
        print(f"Successfully processed {input_image_path} to {output_image_path}")

# -------------------------------------------

preprocessed_images_dir = pathlib.Path("preprocessed_images")
preprocessed_images_dir.mkdir(exist_ok=True)

preprocessed_images_paths = []

for img_path in prescriptions_files_paths:
    output_path = preprocessed_images_dir / img_path.name.replace(".jpg", "_preprocessed.jpg")
    preprocessed_images_paths.append(output_path)
    preprocessing_image(img_path, output_path, {"threshold_value": 127, "use_otsu": True}, save_output=True)
    

In [9]:
# RUNNING MASS OCR ON ALL THE PRESCRIPTION IMAGES

from paddleocr import PaddleOCR
ocr = PaddleOCR(lang='fr')

output_dir = pathlib.Path("ocr_output")
output_dir.mkdir(exist_ok=True)

def ocr_prescriptions(image_paths):
    
    if isinstance(image_paths, (str, pathlib.Path)):
        image_paths = [pathlib.Path(image_paths)]
    else:
        image_paths = [pathlib.Path(p) for p in image_paths]

    ordos = {}

    for path in image_paths:
        key = path.name.removesuffix("_recto.jpg")

        texts = []
        for result in ocr.predict(str(path)):
            texts.extend([t for t in result['rec_texts'] if t.strip()])

        ordos[key] = texts

        output_file = output_dir / f"{key}_ocr.txt"
        output_file.write_text("\n".join(texts), encoding="utf-8")
        print(f"Saved: {output_file}")
    
    return ordos

ocr_prescriptions(preprocessed_images_paths)

## Full mass OCR-pipeline

In [ ]:
# FULL OCR-ing process

# Install required packages
!pip install gdown --quiet
!python -m pip install paddlepaddle==3.2.0 -i https://www.paddlepaddle.org.cn/packages/stable/cpu/
!python -c "import paddle; print(paddle.__version__)"
!python -m pip install "transformers>=5.4.0"
!python -m pip install "paddleocr[all]"
!paddleocr -h

import gdown
import pathlib
import cv2


#----------------------------------------------------------
# DOWNLOADING THE PRESCRIPTION PICTURES FROM GOOGLE DRIVE
# ---------------------------------------------------------

url = "https://drive.google.com/drive/folders/16FthvW2bPYRqt9PyrMI36Mmgruepr_ap?usp=drive_link"

gdown.download_folder(url, quiet=True, use_cookies=False)

!ls # making sure that 'tier_payant_ordonnances'folder is downloaded and present in the current directory 

BASE_PRESCRIPTION_FOLDER = pathlib.Path("tier_payant_ordonnances")

prescriptions_files_paths = list(BASE_PRESCRIPTION_FOLDER.glob("*recto.jpg"))

print("Total number of prescription files:", len(prescriptions_files_paths))
print("---")

for path in prescriptions_files_paths:
    print(path)


#----------------------------------------------------------
# PREPROCESSING THE IMAGES (binarization)
# ---------------------------------------------------------

def apply_binarization(input_path, threshold_value=127, use_otsu=False):
    
    image = cv2.imread(input_path, cv2.IMREAD_GRAYSCALE)
    
    if image is None:
        raise ValueError(f"Could not open or find the image at {input_path}")
    
    if use_otsu:
        _, binary_image = cv2.threshold(image, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    else:
        _, binary_image = cv2.threshold(image, threshold_value, 255, cv2.THRESH_BINARY)

    return binary_image  

def preprocessing_image(input_image_path, output_image_path, binar_params, save_output=True):
    binary_image = apply_binarization(input_image_path, binar_params.get("threshold_value", 127), binar_params.get("use_otsu", False))
    if save_output:
        cv2.imwrite(output_image_path, binary_image)
        print(f"Successfully processed {input_image_path} to {output_image_path}")

# -------------------------------------------

preprocessed_images_dir = pathlib.Path("preprocessed_images")
preprocessed_images_dir.mkdir(exist_ok=True)

preprocessed_images_paths = []

for img_path in prescriptions_files_paths:
    output_path = preprocessed_images_dir / img_path.name.replace(".jpg", "_preprocessed.jpg")
    preprocessed_images_paths.append(output_path)
    preprocessing_image(img_path, output_path, {"threshold_value": 127, "use_otsu": True}, save_output=True)

#----------------------------------------------------------
# RUNNING MASS OCR ON ALL THE PRESCRIPTION IMAGES
# ---------------------------------------------------------

from paddleocr import PaddleOCR
ocr = PaddleOCR(lang='fr')

output_dir = pathlib.Path("ocr_output")
output_dir.mkdir(exist_ok=True)

def ocr_prescriptions(image_paths):
    
    if isinstance(image_paths, (str, pathlib.Path)):
        image_paths = [pathlib.Path(image_paths)]
    else:
        image_paths = [pathlib.Path(p) for p in image_paths]

    ordos = {}

    for path in image_paths:
        key = path.name.removesuffix("_recto.jpg")

        texts = []
        for result in ocr.predict(str(path)):
            texts.extend([t for t in result['rec_texts'] if t.strip()])

        ordos[key] = texts

        output_file = output_dir / f"{key}_ocr.txt"
        output_file.write_text("\n".join(texts), encoding="utf-8")
        print(f"Saved: {output_file}")
    
    return ordos

ocr_prescriptions(preprocessed_images_paths)

#----------------------------------------------------------
# SAVING THE OCR OUTPUT IN DRIVE
#----------------------------------------------------------

from google.colab import drive
drive.mount('/content/drive')

for x in list(pathlib.Path("ocr_output").glob("*.txt")):
    !cp {x} /content/drive/MyDrive/PFE_MASTER_ESSS/


In [7]:
from google.colab import drive
drive.mount('/content/drive')

for x in list(pathlib.Path("ocr_output").glob("*.txt")):
    !cp {x} /content/drive/MyDrive/PFE_MASTER_ESSS/ordos_ocr_output/


---
## 3. Ollama Inference Helper

In [1]:
!nvidia-smi
!apt-get update && apt-get install -y pciutils
!sudo apt-get install zstd 
!pip install deepeval rich pandas tabulate --quiet
!curl -fsSL https://ollama.com/install.sh | sh

!pkill ollama

import pathlib
import subprocess
import time
import json
import re

!pip install ollama
import ollama

from pydantic import BaseModel
server_process = subprocess.Popen(["ollama", "serve"])

from ollama import Client
import httpx

In [ ]:
#─────────────────────────────────────────────────────────────
# PULLING MODELS
#-─────────────────────────────────────────────────────────────

import os

LLM_STRUCT_BENCH_MODELS = [
    "gemma3:27b",
    "gemma3:12b",

    "qwen3:14b",
    "deepseek-r1:14b",
    "deepseek-ri:7b",

    "phi4-mini:3.8b",
    "phi4:14b",

    "qwen3:1.7b",
    "qwen3:4b",
    "qwen3:8b",

]


MED_MODELS = [
    "medgemma:4b",
    "medgemma:27b"
]

MODELS = [
    # Qwen 1
    "qwen:0.5b", 
    "qwen:1.8b",
    "qwen:4b",
    "qwen:7b",

    # Qwen 2
    "qwen2:0.5b", 
    "qwen2:1.5b",
    "qwen2:7b",

    # Qwen 2.5
    "qwen2.5:0.5b",
    "qwen2.5:1.5b",
    "qwen2.5:3b",
    "qwen2.5:7b",

    # Qwen 3
    "qwen3:0.6b",
    "qwen3:1.7b",
    "qwen3:4b",
    "qwen3:4b-instruct",
    "qwen3:8b",
    
    # Qwen 3.5

    # Qwen 3.6
]

# run "ollama serve" in Colab terminal before executing this cell

for m in LLM_STRUCT_BENCH_MODELS:
    os.system(f"ollama pull {m}")

!ollama list

# ─────────────────────────────────────────────────────────────
# MODEL LIST  (all < 8B parameters, popular on Ollama)
# ─────────────────────────────────────────────────────────────


In [9]:
OLLAMA_BASE_URL = "http://localhost:11434"  # Change if Ollama runs on a different host
OLLAMA_TIMEOUT  = 180   # seconds per inference call

client = Client(host=OLLAMA_BASE_URL, timeout=OLLAMA_TIMEOUT)

model_number = 0

print("Models to be evaluated:")
for model in LLM_STRUCT_BENCH_MODELS:
    model_number += 1
    print(f" - {model}")
print(f"\nTotal models: {model_number}")

In [5]:
#-----------------------------------------
# LOADING OCR OUTPUT TEXTS 
#-----------------------------------------
from google.colab import drive
ocr_output_path = pathlib.Path("ocr_output")

if not ocr_output_path.exists():
    print("OCR output not found locally, fetching from Google Drive...")
    drive.mount('/content/drive')
    %mkdir ocr_output
    %cp /content/drive/MyDrive/PFE_MASTER_ESSS/*.txt ocr_output/
    print("Done.")
else:
    print(f"OCR output found at {ocr_output_path}, skipping Drive fetch.")

ocr_texts = {}
for txt_file in ocr_output_path.glob("*.txt"):
    key = txt_file.stem.removesuffix("_ocr")
    ocr_texts[key] = txt_file.read_text(encoding="utf-8")
print(len(ocr_texts), "OCR outputs loaded.")

In [6]:
def check_models_available(models: list) -> dict:
    """Check which models are actually pulled in Ollama."""
    try:
        # Get the list of models currently on the system
        response = ollama.list()
        print(response)
        
        # Access names using the attribute 'model' or 'name' 
        # depending on your library version
        available = {m.model for m in response.models}
        
    except Exception as e:
        print(f"⚠️  Could not connect to Ollama or parse list: {e}")
        return {m: False for m in models}

    status = {}
    for m in models:
        # Check for exact match or if the requested string is part of the tag
        found = m in available or any(m in a for a in available)
        status[m] = found
        icon = "✅" if found else "❌"
        print(f"  {icon} {m}")
        
    return status

In [ ]:
class MedicalFacility(BaseModel):
    name: str or None
    type: str or None    

class Patient(BaseModel):
    full_name: str or None
    first_name: str or None
    family_name: str or None
    date_of_birth: str or None
    age: int or None
    social_security_number: str or None

class Doctor(BaseModel):
    name: str or None
    specialty: str or None
    license_number: str or None

class Medication(BaseModel):
    name: str
    dosage: str or None
    form: str or None
    posologie: str or None
    qsp: str or None

class PrescriptionFrontData(BaseModel):
    medical_facility: MedicalFacility
    patient: Patient
    prescription_date: str or None
    doctor: Doctor
    medications: list[Medication]

In [7]:
def call_ollama_json(model: str, ocr_text: str, current_iteration: int) -> dict:
    user_msg = USER_TEMPLATE.format(ocr_text=ocr_text)
    t0 = time.time()
    
    # Initialize metadata defaults
    metadata = {
        "model": model,
        "raw_output": "",
        "latency_sec": 0.0,
        "error": None
    }
    parsed_data = None
    
    try:
        response = client.chat(
            model=model,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": user_msg},
            ],
            format="json",
            options={"temperature": 0.0},
        )
        
        metadata["latency_sec"] = time.time() - t0
        raw = response["message"]["content"].strip()
        metadata["raw_output"] = raw

        # Strip markdown fences
        clean = re.sub(r"^```[a-z]*\n?", "", raw)
        clean = re.sub(r"\n?```$", "", clean).strip()

        parsed_data = json.loads(clean)
    
    except httpx.ReadTimeout:
        print("Error: The Ollama request timed out!")
    except json.JSONDecodeError as e:
        metadata["error"] = f"JSON parse error: {e}"
        metadata["latency_sec"] = time.time() - t0
    except Exception as e:
        metadata["error"] = str(e)
        metadata["latency_sec"] = time.time() - t0
        
    # Sanitize model name for filenames (remove special characters)
    safe_name = re.sub(r'[^a-zA-Z0-9]', '_', model)
    
    result_file = f"{safe_name}_result.json"
    metadata_file = f"{safe_name}_metadata.json"

    # to save the results in a folder with the current timestamp and the model name

    #current_time = "t" + str(time.localtime().tm_mday) + "-" + str(time.localtime().tm_mon) + "-" + str(time.localtime().tm_year) + "_"+ str(time.localtime().tm_hour)
    results_folder_name = f'{current_iteration}'
    %mkdir -p {results_folder_name}

    %cd {results_folder_name}
    # 1. Save the parsed response (Only if parsing succeeded)
    with open(result_file, "w", encoding="utf-8") as f:
        json.dump(parsed_data, f, indent=4)

    # 2. Save the meta-data
    with open(metadata_file, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=4)

    %cd ..

    return {
        "result_path": result_file,
        "metadata_path": metadata_file,
        "success": metadata["error"] is None
    }

In [21]:
%pip install ollama --quiet

# ─────────────────────────────────────────────────────────────
# SYSTEM PROMPT and USER TEMPLATE
# ─────────────────────────────────────────────────────────────
SYSTEM_PROMPT = """\
{prompt}
"""
USER_TEMPLATE = """Extract the structured data from this OCR prescription text:
---
{ocr_text}
---"""

'''
def call_ollama_no_save(model: str, ocr_text: str) -> dict:
    """
    Send OCR text to an Ollama model and return:
      - raw_output (str)
      - parsed_json (dict | None)
      - latency_sec (float)
      - error (str | None)
    """
    user_msg = USER_TEMPLATE.format(ocr_text=ocr_text)
    t0 = time.time() # start timer to calculate how long the model takes to respond
    try:
        response = ollama.chat(
            model=model,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": user_msg},
            ],
            options={"temperature": 0.0},  # it's a parameter that defines how "creative" the model can get. a value of 0.0 makes the model deterministic
        )
        latency = time.time() - t0
        raw = response["message"]["content"].strip() # this is to only get the model's response without extra spaces or extra information

        # Strip markdown fences if model wrapped output
        clean = re.sub(r"^```[a-z]*\n?", "", raw)
        clean = re.sub(r"\n?```$", "", clean).strip()

        parsed = json.loads(clean)
        return {"raw_output": raw, "parsed_json": parsed, "latency_sec": latency, "error": None}

    except json.JSONDecodeError as e:
        return {"raw_output": raw if 'raw' in dir() else "", "parsed_json": None,
                "latency_sec": time.time() - t0, "error": f"JSON parse error: {e}"}
    except Exception as e:
        return {"raw_output": "", "parsed_json": None,
                "latency_sec": time.time() - t0, "error": str(e)} 

'''

def call_ollama_wtih_save(model: str, ocr_text: str, prompt: str) -> dict:
    
    user_msg = USER_TEMPLATE.format(ocr_text=ocr_text)
    system_prompt = SYSTEM_PROMPT.format(prompt=prompt)
    t0 = time.time()
    
    metadata = {
        "model": model,
        "raw_output": "",
        "latency_sec": 0.0,
        "error": None
    }
    parsed_data = None
    
    try:
        response = ollama.chat(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": user_msg},
            ],
            format="json", # forces the model to respond in a JSON format
            options={"temperature": 0.0},
        )
        
        metadata["latency_sec"] = time.time() - t0
        raw = response["message"]["content"].strip()
        metadata["raw_output"] = raw

        # Strip markdown fences
        clean = re.sub(r"^```[a-z]*\n?", "", raw)
        clean = re.sub(r"\n?```$", "", clean).strip()

        parsed_data = json.loads(clean)

    except json.JSONDecodeError as e:
        metadata["error"] = f"JSON parse error: {e}"
        metadata["latency_sec"] = time.time() - t0
    except Exception as e:
        metadata["error"] = str(e)
        metadata["latency_sec"] = time.time() - t0

    # --- File Writing Logic ---
    
    # Sanitize model name for filenames (remove special characters)
    safe_name = re.sub(r'[^a-zA-Z0-9]', '_', model)
    
    result_file = f"{safe_name}_result.json"
    metadata_file = f"{safe_name}_metadata.json"

    # to save the results in a folder with the current timestamp and the model name

    current_time = "t" + str(time.localtime().tm_mday) + "-" + str(time.localtime().tm_mon) + "-" + str(time.localtime().tm_year) + "_"+ str(time.localtime().tm_hour) + "-" + str(time.localtime().tm_min)
    results_folder_name = f'{current_time}'
    %mkdir -p {results_folder_name}

    %cd {results_folder_name}
    # 1. Save the parsed response (Only if parsing succeeded)
    with open(result_file, "w", encoding="utf-8") as f:
        json.dump(parsed_data, f, indent=4)

    # 2. Save the meta-data
    with open(metadata_file, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=4)

    %cd ..

    return {
        "result_path": result_file,
        "metadata_path": metadata_file,
        "success": metadata["error"] is None
    }

In [ ]:
print("Checking model availability...")
model_status = check_models_available(LLM_STRUCT_BENCH_MODELS)
AVAILABLE_MODELS = [m for m, ok in model_status.items() if ok]
print(f"\n{len(AVAILABLE_MODELS)}/{len(LLM_STRUCT_BENCH_MODELS)} models ready.")

#-----------------------------------------
# TESTING THE MODELS 
#-----------------------------------------

models_responses = {}

#model = AVAILABLE_MODELS[0]  # Just test the first available model for now
#models_responses[model] = call_ollama_json(model, ocr_texts["ordo1"], 3)  # Quick test call to warm up the model
#if models_responses[model]["success"]:
#    print(f"✅ {model} responded successfully in {models_responses[model]['metadata_path']}")

ITERATION = 3


for model in AVAILABLE_MODELS:
    print(f"Now running {model} ...")
    models_responses[model] = call_ollama_json(model, ocr_texts["ordo1"], ITERATION)  # Quick test call to warm up the model
    if models_responses[model]["success"]:
        print(f"✅ {model} responded successfully in {models_responses[model]['metadata_path']}")


---
## 4. Custom DeepEval Metrics

In [ ]:
from deepeval.metrics import BaseMetric
from deepeval.test_case import LLMTestCase
import math

# ─────────────────────────────────────────────────────────────
# Helper: fuzzy string match (normalise before comparing)
# ─────────────────────────────────────────────────────────────
def normalise(s) -> str: #this function strips all non-alphanumeric characters and lowercases the string for a more accurate comparison
    if s is None:
        return ""
    return re.sub(r"[^a-z0-9]", "", str(s).lower()) 

def str_match(a, b) -> bool: #this one just compares two strings for an exact match after normalisation
    return normalise(a) == normalise(b)

def partial_match(pred, gold) -> float: #this function calculates a similarity score between two strings
    """Score 0–1 for fuzzy string similarity."""
    p, g = normalise(pred), normalise(gold)
    if not g:
        return 1.0 if not p else 0.5   # null expected: null predicted = perfect
    if not p:
        return 0.0
    if p == g:
        return 1.0
    # Longest common subsequence ratio
    from difflib import SequenceMatcher
    return SequenceMatcher(None, p, g).ratio()

# ─────────────────────────────────────────────────────────────
# METRIC 1: JSON Validity
# ─────────────────────────────────────────────────────────────
class JSONValidityMetric(BaseMetric): #this is for checking the validity of the JSON output, it's more of an extra check because the model is already supposed to return valid JSON (bcuz of the format="json" parameter in the call), but it's still important to run this test

    """Did the model return parseable JSON at all?"""

    name = "JSON Validity"
    threshold = 1.0

    def measure(self, test_case: LLMTestCase) -> float:
        try:
            parsed = json.loads(test_case.actual_output)
            self.score = 1.0
            self.reason = "Valid JSON"
        except Exception as e:
            self.score = 0.0
            self.reason = f"Invalid JSON: {e}"
        return self.score

    def is_successful(self) -> bool:
        return self.score >= self.threshold

    async def a_measure(self, test_case: LLMTestCase) -> float:
        return self.measure(test_case)


# ─────────────────────────────────────────────────────────────
# METRIC 2: Schema Compliance
# ─────────────────────────────────────────────────────────────
REQUIRED_KEYS = {"medical_facility", "patient", "prescription_date", "doctor", "medications"}
MEDICALFACILITY_KEYS = {"name", "type"}
PATIENT_KEYS  = {"name", "date_of_birth", "id"}
DOCTOR_KEYS   = {"name", "specialty", "license_number"}
MED_KEYS      = {"name", "dosage", "form", "posologie", "qsp"}

class SchemaComplianceMetric(BaseMetric):
    """Does the JSON match the required schema structure?"""
    
    name = "Schema Compliance"
    threshold = 0.8

    def measure(self, test_case: LLMTestCase) -> float:
        #--------------------------------------------------------------------------------------------------------------------------
        # JSON parsing test (should already be valid if it passed the previous metric, but we need to parse it to check the schema)
        #--------------------------------------------------------------------------------------------------------------------------
        try:
            data = json.loads(test_case.actual_output) #TODO: i'm supposed to remove this because the previous metric already checks for JSON validity, so normalement if it fails the json validity test it shouldn't even run this one
        except Exception:
            self.score = 0.0; self.reason = "Unparseable JSON"; return 0.0

        #--------------------------------------------------------------------------------------------------------------------------
        # Fields check 
        #--------------------------------------------------------------------------------------------------------------------------
        checks = []
        checks.append(isinstance(data, dict)) # checks that the top-level structure is a JSON object
        checks.append(REQUIRED_KEYS.issubset(data.keys())) # checks that all required top-level keys are present
        checks.append(isinstance(data.get("medical_facility"), dict) and MEDICALFACILITY_KEYS.issubset(data["medical_facility"].keys())) # checks that the medical_facility field is a dict and contains the required keys
        checks.append(isinstance(data.get("patient"), dict) and PATIENT_KEYS.issubset(data["patient"].keys())) # checks that the patient field is a dict and contains the required keys
        checks.append(isinstance(data.get("doctor"), dict) and DOCTOR_KEYS.issubset(data["doctor"].keys())) # checks that the doctor field is a dict and contains the required keys
        checks.append(isinstance(data.get("medications"), list)) # checks that the medications field is a list

        #--------------------------------------------------------------------------------------------------------------------------
        # List of medicines check (if medications is a list, check that each item is a dict and contains the required keys)
        #--------------------------------------------------------------------------------------------------------------------------
        meds = data.get("medications", [])
        if meds:
            checks.append(all(MED_KEYS.issubset(m.keys()) for m in meds))

        self.score = sum(checks) / len(checks)
        self.reason = f"{sum(checks)}/{len(checks)} schema checks passed"
        return self.score

    def is_successful(self) -> bool:
        return self.score >= self.threshold

    async def a_measure(self, test_case: LLMTestCase) -> float:
        return self.measure(test_case)


# ─────────────────────────────────────────────────────────────
# METRIC 3: Entity Accuracy  (the core metric)
# ─────────────────────────────────────────────────────────────
class EntityAccuracyMetric(BaseMetric):
    """
    Compares extracted entities against ground truth.
    expected_output must be the ground truth JSON string.
    """
    name = "Entity Accuracy"
    threshold = 0.7

    def measure(self, test_case: LLMTestCase) -> float:
        try:
            pred = json.loads(test_case.actual_output)
        except Exception:
            self.score = 0.0; self.reason = "Unparseable JSON"; return 0.0

        gold = json.loads(test_case.expected_output)
        scores = []

        # Patient fields
        for f in ["name", "date_of_birth", "id"]:
            scores.append(partial_match(
                pred.get("patient", {}).get(f),
                gold.get("patient", {}).get(f)
            ))

        # Prescription date
        scores.append(partial_match(pred.get("prescription_date"), gold.get("prescription_date")))

        # Doctor fields
        for f in ["name", "specialty", "license_number"]:
            scores.append(partial_match(
                pred.get("doctor", {}).get(f),
                gold.get("doctor", {}).get(f)
            ))

        # Medication matching (align by index; penalise missing/extra)
        pred_meds = pred.get("medications", [])
        gold_meds = gold.get("medications", [])
        n = max(len(pred_meds), len(gold_meds))
        med_scores = []
        for i in range(n):
            pm = pred_meds[i] if i < len(pred_meds) else {}
            gm = gold_meds[i] if i < len(gold_meds) else {}
            for f in ["name", "dosage", "posologie", "qsp"]:
                med_scores.append(partial_match(pm.get(f), gm.get(f)))
        if med_scores:
            scores.append(sum(med_scores) / len(med_scores))

        self.score = round(sum(scores) / len(scores), 4)
        self.reason = f"Mean entity match = {self.score:.2%} over {len(scores)} fields"
        return self.score

    def is_successful(self) -> bool:
        return self.score >= self.threshold

    async def a_measure(self, test_case: LLMTestCase) -> float:
        return self.measure(test_case)


# ─────────────────────────────────────────────────────────────
# METRIC 4: Medication Name Recall
# ─────────────────────────────────────────────────────────────
class MedicationRecallMetric(BaseMetric):
    """How many medication names from the ground truth did the model find?"""
    name = "Medication Recall"
    threshold = 0.8

    def measure(self, test_case: LLMTestCase) -> float:
        try:
            pred = json.loads(test_case.actual_output)
            gold = json.loads(test_case.expected_output)
        except Exception:
            self.score = 0.0; self.reason = "Parse error"; return 0.0

        gold_names = [normalise(m["name"]) for m in gold.get("medications", [])]
        pred_names = [normalise(m.get("name", "")) for m in pred.get("medications", [])]

        if not gold_names:
            self.score = 1.0; self.reason = "No medications in ground truth"; return 1.0

        found = sum(
            any(partial_match(pn, gn) > 0.8 for pn in pred_names)
            for gn in gold_names
        )
        self.score = round(found / len(gold_names), 4)
        self.reason = f"Found {found}/{len(gold_names)} medications"
        return self.score

    def is_successful(self) -> bool:
        return self.score >= self.threshold

    async def a_measure(self, test_case: LLMTestCase) -> float:
        return self.measure(test_case)


print("✅ All 4 custom metrics defined:")
print("   1. JSONValidityMetric      — did model return parseable JSON?")
print("   2. SchemaComplianceMetric  — does JSON match required schema?")
print("   3. EntityAccuracyMetric    — how accurate are extracted fields?")
print("   4. MedicationRecallMetric  — how many drug names were found?")

---
## 5. Run Evaluation

## Loading stuff before the evaluation

In [ ]:
#-----------------------------------------------------------------
# LOADING THE SYSTEM PROMPT
#-----------------------------------------------------------------

!cp /content/drive/MyDrive/PFE_MASTER_ESSS/system_prompt.txt .
prompt = pathlib.Path("system_prompt.txt").read_text(encoding="utf-8")

#-----------------------------------------------------------------
# LOADING THE OCR RESULTS TO BE USED IN THE EXAMPLES AND EVALUATION
#-----------------------------------------------------------------

!cp -r /content/drive/MyDrive/PFE_MASTER_ESSS/*.txt ocr_output/
ordos_ocr = {}

for x in pathlib.Path("ocr_output").glob("*.txt"):
    ordos_ocr[str(x).removeprefix("ocr_output/").removesuffix("_recto_preprocessed.jpg_ocr.txt")] = x.read_text(encoding="utf-8")

#-----------------------------------------------------------------
# LOADING THE EXAMPLES OF FRONT ORDOS TO BE USED IN THE EVALUATION
#-----------------------------------------------------------------

EXAMPLES = []
!mkdir EXAMPLES
!cp -r /content/drive/MyDrive/PFE_MASTER_ESSS/EXAMPLES/* EXAMPLES/

for ex in pathlib.Path("EXAMPLES").glob("*.json"):
    with open(ex, "r", encoding="utf-8") as f:
        data = json.load(f)
        data["ocr_text"] = ordos_ocr.get(data.get("id"), "")
        EXAMPLES.append({
            "name": ex.removesuffix("_example.json"),
            "data": data
        })

print(EXAMPLES[0])


#-----------------------------
# LOADING THE MODELS RESPONSES // no need because deepeval is gonna call the models
#-----------------------------

#ITERATION = 3
#
#llm_struct_bench_models_results = {}
#
#for output in pathlib.Path(".").glob(f"{ITERATION}/*_result.json"):
#    llm_struct_bench_models_results[output.stem.replace("_result", "")] = json.loads(output.read_text(encoding="utf-8"))

## THE Evaluation

In [ ]:
from deepeval.test_case import LLMTestCase
from rich.progress import Progress, SpinnerColumn, TextColumn, BarColumn, TimeElapsedColumn
from rich.console import Console

console = Console()

METRICS = [
    JSONValidityMetric(),
    SchemaComplianceMetric(),
    EntityAccuracyMetric(),
    MedicationRecallMetric(),
]

# results[model][example_id] = {metric_name: score, latency, error, ...}
results = {}

total_calls = len(AVAILABLE_MODELS) * len(EXAMPLES)
console.print(f"\n[bold cyan]Starting evaluation: {len(AVAILABLE_MODELS)} models × {len(EXAMPLES)} examples = {total_calls} inference calls[/]\n")

with Progress(
    SpinnerColumn(),
    TextColumn("[bold blue]{task.description}"),
    BarColumn(),
    TextColumn("{task.completed}/{task.total}"),
    TimeElapsedColumn(),
    console=console
) as progress:
    task = progress.add_task("Evaluating...", total=total_calls)

    for model in AVAILABLE_MODELS:
        results[model] = {}
        for ex in EXAMPLES:
            progress.update(task, description=f"{model} | {ex['id']}")

            # 1. Inference
            infer = call_ollama_with_save(model, ex["ocr_text"], prompt)

            # 2. Build test case
            actual = infer["raw_output"] if infer["parsed_json"] is None else json.dumps(infer["parsed_json"])
            test_case = LLMTestCase(
                input=USER_TEMPLATE.format(ocr_text=ex["ocr_text"]),
                actual_output=actual,
                expected_output=json.dumps(ex["ground_truth"]),
            )

            # 3. Score each metric
            metric_scores = {}
            for metric in METRICS:
                metric.measure(test_case)
                metric_scores[metric.name] = round(metric.score, 4)

            results[model][ex["id"]] = {
                "scores": metric_scores,
                "latency": round(infer["latency_sec"], 2),
                "error": infer["error"],
                "parsed": infer["parsed_json"],
                "raw": infer["raw_output"][:300],  # Truncate for storage
            }
            progress.advance(task)

console.print("\n[bold green]✅ Evaluation complete![/]")

---
## 6. Results — Per-Model Summary Table

In [ ]:
import pandas as pd

METRIC_NAMES = [m.name for m in METRICS]

rows = []
for model in AVAILABLE_MODELS:
    model_results = results[model]
    n = len(model_results)
    avg_scores = {mn: 0.0 for mn in METRIC_NAMES}
    avg_latency = 0.0
    errors = 0

    for ex_id, res in model_results.items():
        for mn in METRIC_NAMES:
            avg_scores[mn] += res["scores"].get(mn, 0.0)
        avg_latency += res["latency"]
        if res["error"]:
            errors += 1

    row = {"Model": model}
    for mn in METRIC_NAMES:
        row[mn] = round(avg_scores[mn] / n, 3)
    row["Avg Latency (s)"] = round(avg_latency / n, 2)
    row["Errors"] = errors
    # Overall score = mean of all metrics (excluding latency/errors)
    row["Overall"] = round(sum(row[mn] for mn in METRIC_NAMES) / len(METRIC_NAMES), 3)
    rows.append(row)

df = pd.DataFrame(rows).sort_values("Overall", ascending=False).reset_index(drop=True)
df.index += 1  # 1-based ranking
df.index.name = "Rank"

# Colour-coded display
def highlight(val, threshold=0.7):
    if isinstance(val, float):
        if val >= 0.9: color = "background-color: #2d6a4f; color: white"
        elif val >= 0.7: color = "background-color: #52b788"
        elif val >= 0.5: color = "background-color: #f4a261"
        else: color = "background-color: #e63946; color: white"
        return color
    return ""

styled = df.style.applymap(highlight, subset=METRIC_NAMES + ["Overall"])
styled

---
## 7. Per-Example Breakdown

In [ ]:
# Pivot: for a chosen metric, show all models × all examples
TARGET_METRIC = "Entity Accuracy"  # ← Change to any metric name

pivot_rows = []
for model in AVAILABLE_MODELS:
    row = {"Model": model}
    for ex in EXAMPLES:
        score = results[model][ex["id"]]["scores"].get(TARGET_METRIC, None)
        row[ex["id"]] = round(score, 3) if score is not None else "ERR"
    pivot_rows.append(row)

pivot_df = pd.DataFrame(pivot_rows).set_index("Model")
print(f"\n📊 {TARGET_METRIC} — per-example scores:")
pivot_df.style.applymap(highlight, subset=list(pivot_df.columns))

---
## 8. Visualisations

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ── Plot 1: Overall score bar chart ───────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

colors = plt.cm.RdYlGn(df["Overall"].values)
bars = axes[0].barh(df["Model"], df["Overall"], color=colors, edgecolor="white", height=0.6)
axes[0].set_xlim(0, 1)
axes[0].set_xlabel("Score", fontsize=11)
axes[0].set_title("Overall Score (avg of all metrics)", fontsize=13, fontweight="bold")
axes[0].axvline(0.7, color="orange", linestyle="--", linewidth=1, label="threshold 0.7")
axes[0].legend()
for bar, val in zip(bars, df["Overall"]):
    axes[0].text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                 f"{val:.2f}", va="center", fontsize=9)

# ── Plot 2: Latency vs Entity Accuracy scatter ─────────────────
axes[1].scatter(df["Avg Latency (s)"], df["Entity Accuracy"],
                c=df["Overall"], cmap="RdYlGn", s=120, edgecolors="black", linewidths=0.8)
for _, row_ in df.iterrows():
    axes[1].annotate(row_["Model"].split(":")[0],
                     (row_["Avg Latency (s)"], row_["Entity Accuracy"]),
                     textcoords="offset points", xytext=(5, 5), fontsize=8)
axes[1].set_xlabel("Avg Latency (s)", fontsize=11)
axes[1].set_ylabel("Entity Accuracy", fontsize=11)
axes[1].set_title("Quality vs Speed Trade-off", fontsize=13, fontweight="bold")
axes[1].set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig("/tmp/eval_plots1.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Plot 3: Radar / spider chart per model ─────────────────────
from matplotlib.patches import FancyArrowPatch

RADAR_METRICS = METRIC_NAMES  # all 4 metrics
N = len(RADAR_METRICS)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # close the polygon

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
cmap = plt.cm.tab10

for i, model in enumerate(AVAILABLE_MODELS):
    vals = [df[df["Model"] == model][mn].values[0] for mn in RADAR_METRICS]
    vals += vals[:1]
    ax.plot(angles, vals, linewidth=1.8, color=cmap(i), label=model)
    ax.fill(angles, vals, alpha=0.08, color=cmap(i))

ax.set_xticks(angles[:-1])
ax.set_xticklabels(RADAR_METRICS, fontsize=10)
ax.set_ylim(0, 1)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_title("Model Capability Radar", fontsize=14, fontweight="bold", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.15), fontsize=8)

plt.tight_layout()
plt.savefig("/tmp/eval_plots2.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 9. Inspect Individual Outputs

In [ ]:
from rich.table import Table
from rich.panel import Panel
from rich.syntax import Syntax

# ── Change these to inspect any model/example combination ──────
INSPECT_MODEL   = AVAILABLE_MODELS[0]   # Best model (sorted by Overall)
INSPECT_EXAMPLE = "ex_002"               # The noisy OCR example
# ──────────────────────────────────────────────────────────────

r = results[INSPECT_MODEL][INSPECT_EXAMPLE]
ex = next(e for e in EXAMPLES if e["id"] == INSPECT_EXAMPLE)

console.print(Panel(f"[bold]{INSPECT_MODEL}[/] on [bold]{INSPECT_EXAMPLE}[/] — {ex['description']}",
                    style="cyan"))

# Scores table
t = Table(show_header=True, header_style="bold magenta")
t.add_column("Metric"); t.add_column("Score", justify="right")
for mn, sc in r["scores"].items():
    color = "green" if sc >= 0.7 else "red"
    t.add_row(mn, f"[{color}]{sc:.3f}[/]")
t.add_row("Latency", f"{r['latency']}s")
if r["error"]:
    t.add_row("Error", f"[red]{r['error']}[/]")
console.print(t)

# Side-by-side JSON diff
console.print("\n[bold]Model output:[/]")
if r["parsed"]:
    console.print(Syntax(json.dumps(r["parsed"], indent=2, ensure_ascii=False), "json", theme="monokai"))
else:
    console.print(f"[red]Raw output (not valid JSON):[/]\n{r['raw']}")

console.print("\n[bold]Ground truth:[/]")
console.print(Syntax(json.dumps(ex["ground_truth"], indent=2, ensure_ascii=False), "json", theme="monokai"))

---
## 10. Export Results to CSV

In [ ]:
# ── Summary CSV ────────────────────────────────────────────────
df.to_csv("ocr_eval_summary.csv")
print("Saved: ocr_eval_summary.csv")

# ── Detailed CSV (every model × example × metric) ─────────────
detail_rows = []
for model in AVAILABLE_MODELS:
    for ex in EXAMPLES:
        r = results[model][ex["id"]]
        row = {"model": model, "example_id": ex["id"], "description": ex["description"],
               "latency_s": r["latency"], "error": r["error"]}
        row.update(r["scores"])
        detail_rows.append(row)

detail_df = pd.DataFrame(detail_rows)
detail_df.to_csv("ocr_eval_detailed.csv", index=False)
print("Saved: ocr_eval_detailed.csv")

# ── Print final ranking ────────────────────────────────────────
console.print("\n[bold cyan]🏆 Final Ranking[/]")
for rank, (_, row_) in enumerate(df.iterrows(), 1):
    medal = ["🥇", "🥈", "🥉"].get(rank - 1, "  ")
    console.print(f"{medal} #{rank:2d}  [bold]{row_['Model']:<30}[/]  Overall: {row_['Overall']:.3f}  "
                  f"Entity Accuracy: {row_['Entity Accuracy']:.3f}  "
                  f"Latency: {row_['Avg Latency (s)']}s")

---
## 11. Add Your Own Examples

To extend the benchmark, append to `EXAMPLES` in Cell 3 following this template:

```python
{
    "id": "ex_006",
    "description": "Your description",
    "ocr_text": """< paste raw OCR here >""",
    "ground_truth": {
        "patient": {"name": "...", "date_of_birth": "YYYY-MM-DD", "id": None},
        "prescription_date": "YYYY-MM-DD",
        "doctor": {"name": "...", "specialty": None, "license_number": None},
        "medications": [
            {"name": "...", "dosage": "...", "form": "...",
             "frequency": "...", "duration": None, "quantity": None}
        ],
        "diagnoses": ["..."],
        "refills": None,
        "notes": None
    }
}
```

To add your own models, append to `MODELS` and run `ollama pull <model_name>` first.

---
## 12. Tips & Troubleshooting

| Issue | Fix |
|---|---|
| Model not found | Run `ollama pull <model>` in terminal |
| Timeout errors | Increase `OLLAMA_TIMEOUT` or use a smaller model |
| All JSON Validity = 0 | The model may need `format: "json"` — add `"format": "json"` to `ollama.chat()` options |
| Ollama not running | Start with `ollama serve` in terminal |
| Want LLM-based eval | Replace custom metrics with DeepEval's `AnswerRelevancyMetric` + set `DEEPEVAL_TELEMETRY_OPT_OUT=YES` |
